In [1]:
import pandas as pd
import datetime as dt

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
import datetime as dt
from collections import defaultdict

if "NOW" not in globals():
    NOW = dt.datetime(2025, 12, 31)

records = [
    {"CustomerID": 1, "InvoiceDate": dt.datetime(2024, 1, 1), "InvoiceNo": 1001, "TotalAmount": 120.0},
    {"CustomerID": 1, "InvoiceDate": dt.datetime(2024, 1, 10), "InvoiceNo": 1002, "TotalAmount": 85.0},
    {"CustomerID": 2, "InvoiceDate": dt.datetime(2024, 2, 1), "InvoiceNo": 1003, "TotalAmount": 200.0},
    {"CustomerID": 2, "InvoiceDate": dt.datetime(2024, 2, 15), "InvoiceNo": 1004, "TotalAmount": 150.0},
    {"CustomerID": 3, "InvoiceDate": dt.datetime(2024, 3, 1), "InvoiceNo": 1005, "TotalAmount": 95.0},
]

summary = defaultdict(lambda: {"last_invoice_date": None, "frequency": 0, "monetary": 0.0})
for record in records:
    customer_id = record["CustomerID"]
    customer_summary = summary[customer_id]
    customer_summary["frequency"] += 1
    customer_summary["monetary"] += record["TotalAmount"]
    if customer_summary["last_invoice_date"] is None or record["InvoiceDate"] > customer_summary["last_invoice_date"]:
        customer_summary["last_invoice_date"] = record["InvoiceDate"]

rfmTable = []
for customer_id, customer_summary in summary.items():
    rfmTable.append(
        {
            "CustomerID": customer_id,
            "recency": (NOW - customer_summary["last_invoice_date"]).days,
            "frequency": customer_summary["frequency"],
            "monetary": customer_summary["monetary"],
        }
    )

rfmTable


[{'CustomerID': 1, 'recency': 721, 'frequency': 2, 'monetary': 205.0},
 {'CustomerID': 2, 'recency': 685, 'frequency': 2, 'monetary': 350.0},
 {'CustomerID': 3, 'recency': 670, 'frequency': 1, 'monetary': 95.0}]

In [3]:
# Computing RFM Scores via Pandas
import pandas as pd
import datetime as dt

df = pd.DataFrame(records)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

analysis_now = NOW.to_pydatetime() if hasattr(NOW, "to_pydatetime") else NOW

rfmTable = (
    df.groupby("CustomerID").agg(
        recency=("InvoiceDate", lambda x: (analysis_now - x.max()).days),
        frequency=("InvoiceNo", "count"),
        monetary=("TotalAmount", "sum"),
    )
    .reset_index()
    .sort_values("CustomerID")
    .reset_index(drop=True)
)

rfmTable

,CustomerID,recency,frequency,monetary
0,1,721,2,205.0
1,2,685,2,350.0
2,3,670,1,95.0


In [5]:
print(type(records))
print(type(records[0]))
print(records[0])
print([type(k) for k in records[0].keys()])

<class 'list'>
<class 'dict'>
{'CustomerID': 1, 'InvoiceDate': datetime.datetime(2024, 1, 1, 0, 0), 'InvoiceNo': 1001, 'TotalAmount': 120.0}
[<class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>]


In [ ]:
# Implementing K-Means Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Normalize the data (use only numeric RFM columns)
scaler = StandardScaler()
rfm_numeric = rfmTable[["recency", "frequency", "monetary"]]
rfm_normalized = scaler.fit_transform(rfm_numeric)

# Using the Elbow Method to find the optimal number of clusters
wcss = []
n_samples = rfm_numeric.shape[0]
max_k = min(10, n_samples)
for i in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(rfm_normalized)
    wcss.append(kmeans.inertia_)

# Fit the optimal K-Means model
k_opt = min(4, n_samples)
kmeans_final = KMeans(n_clusters=k_opt, init='k-means++', random_state=42)
kmeans_final.fit(rfm_normalized)

rfmTable['Cluster_ID'] = kmeans_final.labels_
rfmTable

/Users/abhishekjadhav/no/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/abhishekjadhav/no/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/abhishekjadhav/no/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/Users/abhishekjadhav/no/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


,CustomerID,recency,frequency,monetary,Cluster_ID
0,1,721,2,205.0,2
1,2,685,2,350.0,1
2,3,670,1,95.0,0
